# Final Fingerprint Project (Colab Notebook)

This notebook processes fingerprint photos placed in the **`pic/`** folder.

It performs these steps for each image:

1. Read image  
2. Convert to grayscale  
3. Remove noise  
4. Enhance contrast  
5. Sharpen / clarify ridge lines  
6. Segment fingerprint region  
7. Apply Sobel edge detection  
8. Apply Canny edge detection  
9. Thin ridges to skeleton  
10. Detect simple minutiae candidates: ridge endings and bifurcations

**How to use**
- Open this notebook in Google Colab.
- Upload fingerprint photos into the `pic/` folder or use the upload cell.
- Run the cells from top to bottom.


In [1]:
# Imports
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print("Imports loaded.")

Imports loaded.


In [2]:
# Folder settings
INPUT_DIR = "pic"
OUTPUT_DIR = "output"

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input folder:", os.path.abspath(INPUT_DIR))
print("Output folder:", os.path.abspath(OUTPUT_DIR))
print("Put your fingerprint photos inside the pic folder.")

Input folder: /Users/islomovic_j/Desktop/DIP_labs/final fingerprint/pic
Output folder: /Users/islomovic_j/Desktop/DIP_labs/final fingerprint/output
Put your fingerprint photos inside the pic folder.


In [3]:
# Optional upload helper for Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Colab.")
    print("Use the next cell if you want to upload files directly.")
else:
    print("Not running in Colab. You can still use local files in the pic folder.")

Not running in Colab. You can still use local files in the pic folder.


In [4]:
# Upload images directly in Colab (optional)
# After upload, files will be saved into the pic folder.

if IN_COLAB:
    uploaded = files.upload()
    for name, data in uploaded.items():
        save_path = os.path.join(INPUT_DIR, name)
        with open(save_path, "wb") as f:
            f.write(data)
    print("Uploaded files saved to pic/")
else:
    print("Skip this cell outside Colab.")

Skip this cell outside Colab.


In [5]:
# Helper functions\n\ndef show_image(title, image, cmap=None, figsize=(6, 6)):\n    plt.figure(figsize=figsize)\n    if image.ndim == 2:\n        plt.imshow(image, cmap=cmap or "gray")\n    else:\n        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))\n    plt.title(title)\n    plt.axis("off")\n    plt.show()\n\n\ndef resize_image(image, target_width=600):\n    h, w = image.shape[:2]\n    if w <= target_width:\n        return image\n    scale = target_width / float(w)\n    new_h = int(h * scale)\n    return cv2.resize(image, (target_width, new_h))\n\n\ndef crop_center_region(image, crop_ratio=0.85):\n    h, w = image.shape[:2]\n    ch = int(h * crop_ratio)\n    cw = int(w * crop_ratio)\n    y1 = max((h - ch) // 2, 0)\n    x1 = max((w - cw) // 2, 0)\n    return image[y1:y1+ch, x1:x1+cw]\n\n\ndef segment_fingerprint_region(gray):\n    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)\n    kernel = np.ones((5, 5), np.uint8)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)\n    return mask\n\n\ndef preprocess_fingerprint(image):\n    steps = {}\n\n    original = image.copy()\n    steps["01_original"] = original\n\n    resized = resize_image(original, 600)\n    steps["02_resized"] = resized\n\n    cropped = crop_center_region(resized, 0.9)\n    steps["03_cropped"] = cropped\n\n    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)\n    steps["04_grayscale"] = gray\n\n    denoised = cv2.GaussianBlur(gray, (5, 5), 0)\n    steps["05_denoised"] = denoised\n\n    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))\n    enhanced = clahe.apply(denoised)\n    steps["06_enhanced"] = enhanced\n\n    sharpen_kernel = np.array([[0, -1, 0],\n                               [-1, 5, -1],\n                               [0, -1, 0]], dtype=np.float32)\n    sharpened = cv2.filter2D(enhanced, -1, sharpen_kernel)\n    steps["07_sharpened"] = sharpened\n\n    seg_mask = segment_fingerprint_region(sharpened)\n    steps["08_segmentation_mask"] = seg_mask\n\n    binary = cv2.adaptiveThreshold(\n        sharpened,\n        255,\n        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,\n        cv2.THRESH_BINARY_INV,\n        15,\n        5\n    )\n\n    segmented = cv2.bitwise_and(binary, seg_mask)\n    steps["09_segmented_binary"] = segmented\n\n    kernel = np.ones((3, 3), np.uint8)\n    cleaned = cv2.morphologyEx(segmented, cv2.MORPH_OPEN, kernel)\n    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)\n    steps["10_cleaned"] = cleaned\n\n    sobelx = cv2.Sobel(sharpened, cv2.CV_64F, 1, 0, ksize=3)\n    sobely = cv2.Sobel(sharpened, cv2.CV_64F, 0, 1, ksize=3)\n    sobel = cv2.magnitude(sobelx, sobely)\n    sobel = np.uint8(np.clip(sobel, 0, 255))\n    steps["11_sobel"] = sobel\n\n    canny = cv2.Canny(sharpened, 50, 150)\n    steps["12_canny"] = canny\n\n    skeleton = cv2.ximgproc.thinning(cleaned)\n    steps["13_skeleton"] = skeleton\n\n    return steps\n\n\ndef detect_minutiae(skeleton):\n    skel = (skeleton > 0).astype(np.uint8)\n    minutiae_img = cv2.cvtColor(skeleton, cv2.COLOR_GRAY2BGR)\n\n    endings = []\n    bifurcations = []\n\n    rows, cols = skel.shape\n\n    for y in range(1, rows - 1):\n        for x in range(1, cols - 1):\n            if skel[y, x] == 1:\n                neighborhood = skel[y-1:y+2, x-1:x+2]\n                neighbors = int(np.sum(neighborhood)) - 1\n\n                if neighbors == 1:\n                    endings.append((x, y))\n                elif neighbors >= 3:\n                    bifurcations.append((x, y))\n\n    for x, y in endings:\n        cv2.circle(minutiae_img, (x, y), 2, (0, 255, 0), -1)\n\n    for x, y in bifurcations:\n        cv2.circle(minutiae_img, (x, y), 2, (0, 0, 255), -1)\n\n    return minutiae_img, endings, bifurcations\n\n\ndef save_step_images(base_name, steps, minutiae_img):\n    sample_dir = os.path.join(OUTPUT_DIR, base_name)\n    os.makedirs(sample_dir, exist_ok=True)\n\n    for step_name, img in steps.items():\n        save_path = os.path.join(sample_dir, f"{step_name}.png")\n        cv2.imwrite(save_path, img)\n\n    cv2.imwrite(os.path.join(sample_dir, "14_minutiae.png"), minutiae_img)\n    return sample_dir\n

In [6]:
# Find images in the pic folder
valid_ext = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
image_files = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_ext)])

print("Images found:", len(image_files))
for f in image_files:
    print("-", f)

if not image_files:
    print("No images found. Upload or place your fingerprint photos in the pic folder first.")

Images found: 7
- 1.jpg
- 2.jpg
- 3.jpg
- 4.jpg
- 5.jpg
- 6.jpg
- 7.jpg


In [7]:
# Process all images and save step-by-step outputs
results_summary = []

for file_name in image_files:
    file_path = os.path.join(INPUT_DIR, file_name)
    image = cv2.imread(file_path)

    if image is None:
        print("Could not read:", file_name)
        continue

    steps = preprocess_fingerprint(image)
    minutiae_img, endings, bifurcations = detect_minutiae(steps["13_skeleton"])

    base_name = os.path.splitext(file_name)[0]
    saved_dir = save_step_images(base_name, steps, minutiae_img)

    results_summary.append({
        "file": file_name,
        "ridge_endings": len(endings),
        "bifurcations": len(bifurcations),
        "saved_dir": saved_dir
    })

    print("=" * 70)
    print("Processed:", file_name)
    print("Saved to:", saved_dir)
    print("Ridge endings:", len(endings))
    print("Bifurcations:", len(bifurcations))

print("\nDone processing all images.")

NameError: name 'preprocess_fingerprint' is not defined

In [ ]:
# Show full step-by-step result for one selected image
if image_files:
    selected_file = image_files[0]   # Change index if you want another image
    print("Showing full pipeline for:", selected_file)

    image = cv2.imread(os.path.join(INPUT_DIR, selected_file))
    steps = preprocess_fingerprint(image)
    minutiae_img, endings, bifurcations = detect_minutiae(steps["13_skeleton"])

    for step_name, img in steps.items():
        show_image(step_name, img)

    show_image("14_minutiae", minutiae_img)
    print("Ridge endings:", len(endings))
    print("Bifurcations:", len(bifurcations))
else:
    print("No images available.")

In [ ]:
# Show a compact summary table
if results_summary:
    df = pd.DataFrame(results_summary)
    display(df)
else:
    print("No processed results yet.")

## Notes

- **Green dots** = ridge ending candidates  
- **Red dots** = bifurcation candidates  
- Because phone macro photos are not scanner-quality fingerprints, minutiae detection may include false points.
- Better lighting, focus, and closer crop usually improve results.
